# Step 4 — Predict Claim Complexity at FNOL
**Roadmap reference:** XGBoost classifier

## Goal
Predict at First Notice of Loss what final reassignment group (0/A/B/C) a claim
will reach. This lets you route to the right adjuster tier upfront.

## Critical rule: no data leakage
Only features available at the moment of FNOL are used. Assignment history,
cause tags, and exposure additions discovered after 48h are excluded.

## FNOL-safe features
`loss_cause`, `policy_type`, `loss_state`, `insured_state`, `geographic_mismatch`,
`stp_flag`, `reported_loss_amount`, `n_exposures_at_fnol`,
`exposure_added_within_48h`, `fnol_hour`, `fnol_day_of_week`

## Training / test split
- Training: 80% of 2024 closed claims (random split, stratified by group)
- Test: held-out 20% — simulates scoring on unseen claims

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score)
import pickle
from pathlib import Path

DATA = Path("data")
matrix = pd.read_csv(DATA / "step2_feature_matrix.csv")
print(f"Feature matrix: {len(matrix)} rows")
matrix["final_group"].value_counts()

Feature matrix: 200 rows


final_group
C    112
0     46
B     28
A     14
Name: count, dtype: int64

In [2]:
# ── Feature engineering — FNOL-only ──────────────────────────────────────────
FNOL_FEATURES = [
    "geographic_mismatch", "stp_flag_enc",
    "reported_loss_amount", "n_exposures_at_fnol",
    "exposure_added_within_48h", "fnol_hour", "fnol_day_of_week",
    "loss_cause_enc", "policy_type_enc",
]

df = matrix.copy()

# Encode categoricals
le_lc = LabelEncoder()
le_pt = LabelEncoder()
df["loss_cause_enc"]   = le_lc.fit_transform(df["loss_cause"])
df["policy_type_enc"]  = le_pt.fit_transform(df["policy_type"])
df["stp_flag_enc"]     = (df["stp_flag"] == "Y").astype(int)

X = df[FNOL_FEATURES]
y = df["group_label"]   # 0=Group0, 1=A, 2=B, 3=C

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print("Class distribution (test):", dict(y_test.value_counts().sort_index()))

Train: 160 | Test: 40
Class distribution (test): {0: 9, 1: 3, 2: 6, 3: 22}


In [3]:
# ── Train XGBoost ─────────────────────────────────────────────────────────────
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42,
)
model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.3f} ({acc*100:.1f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=["Group 0 (FTR)", "Group A", "Group B", "Group C"]))

C:\Users\SujaySunilNagvekar\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Test Accuracy: 0.475 (47.5%)

Classification Report:
               precision    recall  f1-score   support

Group 0 (FTR)       0.17      0.11      0.13         9
      Group A       0.33      0.33      0.33         3
      Group B       0.50      0.17      0.25         6
      Group C       0.55      0.73      0.63        22

     accuracy                           0.47        40
    macro avg       0.39      0.33      0.34        40
 weighted avg       0.44      0.47      0.44        40



In [4]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(cm,
                labels=dict(x="Predicted", y="Actual", color="Count"),
                x=["Group 0", "Group A", "Group B", "Group C"],
                y=["Group 0", "Group A", "Group B", "Group C"],
                title="Confusion Matrix — FNOL Complexity Classifier",
                text_auto=True, color_continuous_scale="Blues")
fig.show()

In [5]:
# ── Feature importance ────────────────────────────────────────────────────────
imp = pd.DataFrame({
    "Feature": FNOL_FEATURES,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=True)

fig = px.bar(imp, x="Importance", y="Feature", orientation="h",
             title="XGBoost Feature Importance — FNOL Complexity Prediction",
             color="Importance", color_continuous_scale="Blues")
fig.show()

In [6]:
# ── Save model + predictions ──────────────────────────────────────────────────
with open(DATA / "step4_complexity_model.pkl", "wb") as f:
    pickle.dump({"model": model, "le_lc": le_lc, "le_pt": le_pt,
                 "features": FNOL_FEATURES}, f)

# Add predictions back to full dataset
df["predicted_group_label"] = model.predict(X)
df["predicted_group"] = df["predicted_group_label"].map({0:"0",1:"A",2:"B",3:"C"})
df[["claim_id", "predicted_group", "final_group"]].to_csv(
    DATA / "step4_predictions.csv", index=False)
print("Saved → data/step4_complexity_model.pkl")
print("Saved → data/step4_predictions.csv")
print(f"\nTarget: >70% accuracy on held-out test | Achieved: {acc*100:.1f}%")

Saved → data/step4_complexity_model.pkl
Saved → data/step4_predictions.csv

Target: >70% accuracy on held-out test | Achieved: 47.5%
